# Módulo 4 — Prompt Engineering para Análise de Dados

Um bom prompt é a diferença entre receber código genérico inutilizável e código que funciona no seu contexto específico.

## O que vamos ver

1. Princípios de prompts eficazes
2. Estrutura de um prompt de análise de dados
3. Few-shot prompting com SQL/pandas
4. Prompts para documentação automática
5. Exemplos ruins vs. bons

---

## 1. Princípios de Prompts Eficazes

Os 5 elementos de um bom prompt para análise de dados:

| Elemento | Descrição | Exemplo |
|----------|-----------|--------|
| **Contexto** | Quem você é e qual é o domínio | "Sou analista de qualidade de dados em uma distribuidora de energia" |
| **Tarefa** | O que precisa ser feito, com verbo claro | "Crie uma função que..." / "Explique por que..." |
| **Dados** | Estrutura dos dados (colunas, tipos) | "DataFrame com colunas: cod_uc (int), consumo_kwh (float)" |
| **Formato** | Como quer a resposta | "Retorne um dict com chaves X, Y, Z" |
| **Restrições** | O que não fazer | "Use only pandas, sem loops explícitos" |

In [ ]:
# Demonstração: estrutura de prompt em Python como string template
# (útil para criar prompts programaticamente)

def criar_prompt_analise_qualidade(
    nome_tabela: str,
    colunas: dict,
    regras_negocio: list,
    formato_retorno: str
) -> str:
    """
    Gera um prompt estruturado para análise de qualidade de dados.
    
    Args:
        nome_tabela: nome da tabela/DataFrame a analisar
        colunas: dict {nome_coluna: tipo_e_descricao}
        regras_negocio: lista de regras de validação
        formato_retorno: descrição do formato esperado
    """
    colunas_str = "\n".join([f"  - {col}: {desc}" for col, desc in colunas.items()])
    regras_str = "\n".join([f"  {i+1}. {r}" for i, r in enumerate(regras_negocio)])
    
    return f"""Contexto:
Sou analista de dados de uma distribuidora de energia elétrica brasileira.
Trabalhamos com Python 3.11, pandas 2.2 e Snowflake.

Tarefa:
Crie uma função Python para verificar a qualidade dos dados da tabela {nome_tabela}.

Estrutura dos dados:
{colunas_str}

Regras de negócio:
{regras_str}

Formato de retorno:
{formato_retorno}

Requisitos técnicos:
- Use type hints (Python 3.11)
- Docstring em português com exemplos
- Trate valores nulos e tipos inesperados
- Sem loops explícitos — use operações pandas vetorizadas
- Não use bibliotecas além de pandas e re (regex)"""


# Exemplo de uso
prompt = criar_prompt_analise_qualidade(
    nome_tabela="CONSUMIDORES_UC",
    colunas={
        "cod_consumidor": "int, chave primária",
        "cpf_cnpj": "str, CPF (11 dígitos) ou CNPJ (14 dígitos)",
        "cod_modalidade_tarifaria": "str, valores: B1, B2, B3, B4, A4, A3a, A3, A2, A1, AS",
        "cod_classe_consumo": "str, valores: RESIDENCIAL, COMERCIAL, INDUSTRIAL, RURAL, PODER PUBLICO",
        "vlr_consumo_medio_kwh": "float, consumo médio mensal em kWh",
        "flg_ativo": "bool, True = UC ativa"
    },
    regras_negocio=[
        "cod_consumidor deve ser único (não nulo)",
        "cpf_cnpj deve ter 11 dígitos (CPF) ou 14 dígitos (CNPJ) após remover formatação",
        "consumidores RESIDENCIAIS só podem ter modalidade B1",
        "vlr_consumo_medio_kwh deve ser >= 0 e <= 1.000.000",
    ],
    formato_retorno="DataFrame pandas com colunas: cod_consumidor, nome_regra, status (OK/FALHA), detalhe"
)

print(prompt)

## 2. Few-Shot Prompting

Few-shot significa dar exemplos no prompt para mostrar o padrão esperado. Muito eficaz para gerar queries SQL ou código com convenções específicas.

In [ ]:
# Template de few-shot para geração de queries SQL
prompt_few_shot_sql = """
Você é um especialista em SQL Snowflake trabalhando com dados de distribuição de energia elétrica.

Convenções do projeto:
- Schema: CADASTRO
- Tabela principal: CONSUMIDORES_UC
- Sempre use CTE para queries complexas
- Alias de tabelas em snake_case
- Comentários em português

Exemplos:

Pergunta: "Quantos consumidores ativos existem por classe?"
SQL:
-- Contagem de consumidores ativos por classe de consumo
SELECT
    cod_classe_consumo,
    COUNT(*) AS qtd_consumidores
FROM CADASTRO.CONSUMIDORES_UC
WHERE flg_ativo = TRUE
GROUP BY cod_classe_consumo
ORDER BY qtd_consumidores DESC;

Pergunta: "Qual o percentual de registros sem CPF/CNPJ?"
SQL:
-- Percentual de registros com CPF/CNPJ ausente
WITH totais AS (
    SELECT
        COUNT(*) AS total,
        COUNT(cpf_cnpj) AS com_documento
    FROM CADASTRO.CONSUMIDORES_UC
)
SELECT
    total,
    com_documento,
    total - com_documento AS sem_documento,
    ROUND((total - com_documento) / total * 100, 2) AS pct_sem_documento
FROM totais;

Pergunta: "Liste os municípios com mais de 5% dos consumidores sem medidor"
SQL:
"""

print("Prompt few-shot preparado.")
print(f"Comprimento do prompt: {len(prompt_few_shot_sql)} caracteres")
print("\n--- Estrutura ---")
print("1. Contexto e papel")
print("2. Convenções do projeto")
print("3. Exemplos (pergunta → SQL)")
print("4. Nova pergunta a responder")

## 3. Prompts para Geração de Documentação

In [ ]:
# Prompt para documentar uma função existente
codigo_para_documentar = '''
def verificar_qualidade_lote(df, limiar=0.8):
    resultados = {}
    for col in df.columns:
        pct = df[col].notna().sum() / len(df)
        resultados[col] = {"pct_preenchido": pct, "ok": pct >= limiar}
    n_ok = sum(1 for v in resultados.values() if v["ok"])
    resultados["_resumo"] = {"colunas_ok": n_ok, "total": len(df.columns)}
    return resultados
'''

prompt_documentar = f"""
Adicione uma docstring completa à seguinte função Python usada em um
sistema de qualidade de dados de cadastro de consumidores de energia elétrica.

A docstring deve conter:
1. Descrição de uma linha do que a função faz
2. Descrição detalhada (2-3 frases explicando o propósito no contexto do setor)
3. Seção Args com tipo e descrição de cada parâmetro
4. Seção Returns com tipo e estrutura do retorno
5. Exemplo de uso com dados fictícios de energia
6. Raises (se aplicável)

Idioma: Português brasileiro.
Estilo: Google style docstring.

Código:
{codigo_para_documentar}
"""

print("Prompt de documentação:")
print(prompt_documentar)

## 4. Comparação: Prompts Ruins vs. Bons

In [ ]:
comparacoes = [
    {
        "cenario": "Gerar query de qualidade",
        "prompt_ruim": "faça uma query para ver a qualidade dos dados",
        "problema": "Vago demais. Que dados? Que banco? Que é 'qualidade'?",
        "prompt_bom": """Crie uma query SQL Snowflake para a tabela CADASTRO.CONSUMIDORES_UC que:
1. Calcule o % de nulos por coluna (cpf_cnpj, num_medidor, dat_ultima_leitura)
2. Agrupe os resultados por cod_uf
3. Retorne apenas UFs onde algum campo tem > 5% de nulos
Use CTE e nomeie o resultado como 'relatorio_nulos'. Comentários em português.""",
        "motivo": "Especifica tabela, colunas, lógica, agrupamento, filtro e formato"
    },
    {
        "cenario": "Tratar erro de código",
        "prompt_ruim": "meu código não funciona, o que fazer?",
        "problema": "Sem código, sem mensagem de erro, sem contexto",
        "prompt_bom": """Estou recebendo o seguinte erro ao executar este código Python com pandas 2.2:

Erro: ValueError: Cannot convert non-finite values (NA or inf) to integer

Código:
df['cod_consumidor'] = df['cod_consumidor'].astype(int)

O DataFrame tem 50 linhas e a coluna 'cod_consumidor' pode ter valores NaN.
Como corrigir mantendo os registros com NaN?""",
        "motivo": "Inclui erro exato, código, contexto de dados e resultado esperado"
    },
    {
        "cenario": "Explicar conceito",
        "prompt_ruim": "explique DEC",
        "problema": "Pode trazer definição genérica sem aplicação prática",
        "prompt_bom": """Explique o indicador DEC (Duração Equivalente de Interrupção por Consumidor) 
para um analista de dados que nunca trabalhou no setor elétrico. 
Inclua: definição, fórmula, unidade, o que significa um valor alto/baixo, 
e como seria calculado em Python usando pandas a partir de uma tabela de interrupções 
com colunas: id_interrupcao, duracao_horas, qtd_ucs_afetadas, total_ucs_conjunto.""",
        "motivo": "Define o público-alvo, o nível de detalhe e pede aplicação prática"
    }
]

for comp in comparacoes:
    print(f"\n{'='*65}")
    print(f"Cenário: {comp['cenario']}")
    print(f"\nRUIM: '{comp['prompt_ruim']}'")
    print(f"Problema: {comp['problema']}")
    print(f"\nBOM:")
    print(comp['prompt_bom'][:200] + "..." if len(comp['prompt_bom']) > 200 else comp['prompt_bom'])
    print(f"\nMotivo da melhora: {comp['motivo']}")

## 5. Templates de Prompt Reutilizáveis

In [ ]:
# Biblioteca de templates de prompt para o setor elétrico
TEMPLATES_PROMPT = {
    "gerar_validacao": """
Contexto: Analista de qualidade de dados em distribuidora de energia elétrica.
Stack: Python 3.11, pandas 2.2, Snowflake.

Crie uma função de validação para a regra:
[DESCREVER A REGRA]

DataFrame de entrada: df com colunas [LISTAR COLUNAS E TIPOS]

Retorno esperado: Series booleana onde True = registro inválido
Use type hints, docstring em português, sem loops explícitos.
""",
    
    "gerar_query_snowflake": """
Snowflake SQL para schema CADASTRO, tabela CONSUMIDORES_UC.
Colunas principais: cod_consumidor, cpf_cnpj, cod_classe_consumo,
cod_modalidade_tarifaria, nom_municipio, cod_uf, vlr_consumo_medio_kwh,
flg_ativo, dat_ultima_leitura.

Crie uma query que:
[DESCREVER O QUE A QUERY DEVE FAZER]

Requisitos:
- Use CTE quando tiver mais de 2 passos
- Comentários em português
- Ordene o resultado por [CAMPO]
- Limite a [N] registros
""",

    "explicar_resultado": """
Estou analisando dados de qualidade de cadastro de consumidores de energia.
Recebi o seguinte resultado: [COLAR RESULTADO]

Contexto: [DESCREVER DE ONDE VIE ESSE DADO]

Explique:
1. O que esse resultado significa para o negócio
2. Qual é o nível de severidade do problema (crítico/atenção/ok)
3. Que ações deveriam ser tomadas
4. Como priorizar a correção
"""
}

print("Templates disponíveis:")
for nome, template in TEMPLATES_PROMPT.items():
    linhas = template.strip().split("\n")
    print(f"\n[{nome}] — {len(linhas)} linhas")
    print(f"  Uso: preencha os campos entre [COLCHETES]")

print("\n\nExemplo de uso do template 'gerar_query_snowflake':")
print("Substitua [DESCREVER O QUE A QUERY DEVE FAZER] por:")
print("'Mostre os 10 municípios com maior % de consumidores sem CPF/CNPJ, entre os ativos'")